# GETTSIM Workshop 2026

## Stage 1 — The policy environment

This notebook is supposed to serve as a template for your own reform.

Here, we analyse an example reform: Raising the rate at which earnings between 520 € and
1000 € per month are withdrawn from Bürgergeld, so the retained share on that band falls
from 30 % to 15 %.

In [ ]:
import numpy as np
import pandas as pd
from types import ModuleType

from gettsim import (
    InputData,
    MainTarget,
    TTTargets,
    copy_environment,
    main,
)
from gettsim.tt import (
    PiecewisePolynomialParam,
    PiecewisePolynomialParamValue,
    ScalarParam,
    piecewise_polynomial,
    policy_function,
    TTSIMUnit,
    get_piecewise_parameters,
)

POLICY_DATE = "2023-07-01"

### 1. The policy environment

`main` returns the nested dictionary of every parameter and function that holds on a
given date.

In [ ]:
status_quo = main(
    main_target=MainTarget.policy_environment,
    policy_date_str=POLICY_DATE,
)

sorted(status_quo) # The namespaces

In [ ]:
status_quo # the entire policy environment

`bürgergeld` is the namespace this notebook works in.

In [ ]:
sorted(status_quo["bürgergeld"])

The leaf we are after is the Freibetrag schedule: how much earned income stays with the
household.

In [ ]:
freibetrag = status_quo["bürgergeld"][
    "parameter_anrechnungsfreies_einkommen_ohne_kinder_in_bg"
]
type(freibetrag)

The Freibetrag is stored as a `PiecewisePolynomialParam`. For an explanation, see the
GETTSIM documentation.

In [ ]:
freibetrag.value

Between 100 € and 520 € of monthly earnings, 20 % of each additional euro stays with the
household; between 520 € and 1000 €, 30 % does. That middle band is the one we change.

Our target is the Bürgergeld claim.

In [ ]:
TARGETS = {"bürgergeld": {"betrag_m_bg": None}}

### 2. Running GETTSIM once

First, we find out which inputs we need.

In [ ]:
template = main(
    main_target=MainTarget.templates.input_data_dtypes.tree,
    policy_date_str=POLICY_DATE,
    tt_targets=TTTargets.tree(TARGETS),
    include_warn_nodes=False,
)

template

Then, we invent some generic inputs, just to see whether GETTSIM computes results
for our reform. This one is a married couple with one child born in 2017: one earner
at 1000 € per month, 600 € cold rent, 65 m². Tomorrow you will not type this by
hand.

In [ ]:
INPUT_DATA_TREE = {
    "alter": np.array([30, 30, 6]),
    "alter_monate": np.array([360, 360, 72]),
    "arbeitsstunden_w": np.array([15, 0, 0]),
    "behinderungsgrad": np.array([0, 0, 0]),
    "geburtsjahr": np.array([1993, 1993, 2017]),  # the child is born in 2017
    "hh_id": np.array([0, 0, 0]),  # all three in one household
    "p_id": np.array([0, 1, 2]),  # two adults and a child
    "vermögen": np.array([0, 0, 0]),
    "wohnort_ost_hh": np.array([False, False, False]),
    "bürgergeld": {
        "bezug_im_vorjahr": np.array([False, False, False]),
        "p_id_einstandspartner": np.array([1, 0, -1]),
    },
    "einkommensteuer": {
        "gemeinsam_veranlagt": np.array([True, True, False]),
        "abzüge": {
            "beitrag_private_rentenversicherung_m": np.array([0, 0, 0]),
            "kinderbetreuungskosten_m": np.array([0, 0, 100]),
            "p_id_kinderbetreuungskostenträger": np.array([-1, -1, 0]),
        },
        "einkünfte": {
            "ist_hauptberuflich_selbstständig": np.array([False, False, False]),
            "aus_forst_und_landwirtschaft": {
                "betrag_y": np.array([0, 0, 0]),
            },
            "aus_gewerbebetrieb": {
                "betrag_y": np.array([0, 0, 0]),
            },
            "aus_nichtselbstständiger_arbeit": {
                "tatsächliche_werbungskosten_y": np.array([0, 0, 0]),
            },
            "aus_selbstständiger_arbeit": {
                "betrag_y": np.array([0, 0, 0]),
            },
            "aus_vermietung_und_verpachtung": {
                "betrag_y": np.array([0, 0, 0]),
            },
            "sonstige": {
                "alle_weiteren_y": np.array([0, 0, 0]),
                "rente": {
                    "steuerpflichtige_einnahmen_m": np.array([0, 0, 0]),
                },
            },
        },
    },
    "einnahmen": {
        "bruttolohn_m": np.array([1000, 0, 0]),  # one earner, 1000 € per month
        "kapitalerträge_y": np.array([0.0, 0.0, 0.0]),
        "renten": {
            "aus_berufsständischen_versicherungen_m": np.array([0, 0, 0]),
            "basisrente_m": np.array([0, 0, 0]),
            "betriebliche_altersvorsorge_m": np.array([0, 0, 0]),
            "geförderte_private_vorsorge_m": np.array([0, 0, 0]),
            "gesetzliche_m": np.array([0, 0, 0]),
            "sonstige_private_vorsorge_m": np.array([0, 0, 0]),
        },
    },
    "elterngeld": {
        "betrag_m": np.array([0, 0, 0]),
    },
    "familie": {
        "alleinerziehend": np.array([False, False, False]),
        "p_id_ehepartner": np.array([1, 0, -1]),  # -1 means nobody
        "p_id_elternteil_1": np.array([-1, -1, 0]),  # the child's parents
        "p_id_elternteil_2": np.array([-1, -1, 1]),
    },
    "grundsicherung": {
        "im_alter": {
            "überschusseinkommen_m_eg": np.array([0, 0, 0]),
        },
    },
    "kindergeld": {
        "in_ausbildung": np.array([False, False, False]),
        "p_id_empfänger": np.array([-1, -1, 0]),
    },
    "sozialversicherung": {
        "arbeitslosen": {
            "arbeitssuchend": np.array([False, False, False]),
            "mean_nettoeinkommen_in_12_monaten_vor_arbeitslosigkeit_m": (
                np.array([0, 0, 0])
            ),
            "monate_beitragspflichtig_versichert_in_letzten_30_monaten": (
                np.array([0, 0, 0])
            ),
            "monate_durchgängigen_bezugs_von_arbeitslosengeld": np.array([0, 0, 0]),
            "monate_sozialversicherungspflichtiger_beschäftigung_in_letzten_5_jahren": (
                np.array([0, 0, 0])
            ),
        },
        "kranken": {
            "beitrag": {
                "bemessungsgrundlage_rente_m": np.array([0, 0, 0]),
                "privat_versichert": np.array([False, False, False]),
            },
        },
        "pflege": {
            "beitrag": {
                "hat_kinder": np.array([True, True, False]),
            },
        },
        "rente": {
            "bezieht_rente": np.array([False, False, False]),
            "grundrente": {
                "grundrentenzeiten_monate": np.array([0, 0, 0]),
            },
        },
    },
    "unterhalt": {
        "tatsächlich_erhaltener_betrag_m": np.array([0, 0, 0]),
    },
    "unterhaltsvorschuss": {
        "betrag_m": np.array([0, 0, 0]),
    },
    "wohnen": {
        "bewohnt_eigentum_hh": np.array([False, False, False]),
        "bruttokaltmiete_m_hh": np.array([600, 600, 600]),  # 600 € cold rent
        "heizkosten_m_hh": np.array([50, 50, 50]),
        "wohnfläche_hh": np.array([65, 65, 65]),  # 65 m²
    },
    "wohngeld": {
        "mietstufe_hh": np.array([5, 5, 5]),
    },
}

In [ ]:
baseline = main(
    main_target=MainTarget.results.df_with_nested_columns,
    policy_date_str=POLICY_DATE,
    input_data=InputData.tree(INPUT_DATA_TREE),
    tt_targets=TTTargets.tree(TARGETS),
    include_warn_nodes=False,
)
baseline

### 3. Changing a parameter

Three steps: copy the environment, build the new parameter object, put it back.

In [ ]:
reform = copy_environment(status_quo)

`get_piecewise_parameters` rebuilds a piecewise schedule from a list of intervals. On
the band between 520 € and 1000 € the retained share goes from 30 % to 15 %, a
withdrawal rate of 85 % instead of 70 %.

There are two schedules: GETTSIM applies the one with children if children live in
the Bedarfsgemeinschaft, the one without children otherwise. Our household has a
child, so changing only the childless schedule would leave its Bürgergeld untouched.
The reform changes both.

In [ ]:
# The two schedules differ only in where the Freibetrag stops: 1200 € without children
# in the Bedarfsgemeinschaft, 1500 € with them.
reform["bürgergeld"]["parameter_anrechnungsfreies_einkommen_ohne_kinder_in_bg"] = (
    PiecewisePolynomialParam(
        value=get_piecewise_parameters(
            func_type="piecewise_linear",
            parameter_list=[
                {"interval": "(-inf, 0)", "intercept": 0, "slope": 0},
                {"interval": "[0, 100)", "slope": 1.0},
                {"interval": "[100, 520)", "slope": 0.2},
                {"interval": "[520, 1000)", "slope": 0.15},  # was 0.3
                {"interval": "[1000, 1200)", "slope": 0.1},
                {"interval": "[1200, inf)", "slope": 0.0},
            ],
            leaf_name="parameter_anrechnungsfreies_einkommen_ohne_kinder_in_bg",
            xnp=np,
        ),
        input_unit=TTSIMUnit.EUR.PER_MONTH,
        output_unit=TTSIMUnit.EUR.PER_MONTH,
    )
)

reform["bürgergeld"]["parameter_anrechnungsfreies_einkommen_mit_kindern_in_bg"] = (
    PiecewisePolynomialParam(
        value=get_piecewise_parameters(
            func_type="piecewise_linear",
            parameter_list=[
                {"interval": "(-inf, 0)", "intercept": 0, "slope": 0},
                {"interval": "[0, 100)", "slope": 1.0},
                {"interval": "[100, 520)", "slope": 0.2},
                {"interval": "[520, 1000)", "slope": 0.15},  # was 0.3
                {"interval": "[1000, 1500)", "slope": 0.1},
                {"interval": "[1500, inf)", "slope": 0.0},
            ],
            leaf_name="parameter_anrechnungsfreies_einkommen_mit_kindern_in_bg",
            xnp=np,
        ),
        input_unit=TTSIMUnit.EUR.PER_MONTH,
        output_unit=TTSIMUnit.EUR.PER_MONTH,
    )
)

Hand the modified environment to `main` as `policy_environment=` and compare.

In [ ]:
after = main(
    main_target=MainTarget.results.df_with_nested_columns,
    policy_date_str=POLICY_DATE,
    policy_environment=reform,
    input_data=InputData.tree(INPUT_DATA_TREE),
    tt_targets=TTTargets.tree(TARGETS),
    include_warn_nodes=False,
)

pd.DataFrame(
    {
        "status quo": baseline.to_numpy().ravel(),
        "reform": after.to_numpy().ravel(),
    }
)

Other parameter classes are built differently; `gettsim/docs/how_to_guides/modifications_of_policy_environments.ipynb`
has a worked example of each.

## 4. Changing a function

GETTSIM picks between two Freibetrag schedules depending on whether children live in the
Bedarfsgemeinschaft. Making the more generous one apply only from the second child needs
a new function and a new parameter. First read the function we are about to replace.

In [ ]:
import inspect

print(
    inspect.getsource(
        status_quo["bürgergeld"]["anrechnungsfreies_einkommen_m"].function
    )
)

The replacement carries the same name, signature and date range. Arguments are other
nodes, addressed by qname — a double underscore separates namespaces.

In [ ]:
@policy_function(
    start_date="2023-01-01",
    unit=TTSIMUnit.CURRENCY.PER_MONTH,
)
def anrechnungsfreies_einkommen_m(
    einnahmen__bruttolohn_m: float,
    einkommensteuer__einkünfte__aus_selbstständiger_arbeit__betrag_m: float,
    familie__anzahl_kinder_bis_17_bg: int,
    parameter_anrechnungsfreies_einkommen_ohne_kinder_in_bg: PiecewisePolynomialParamValue,
    parameter_anrechnungsfreies_einkommen_mit_kindern_in_bg: PiecewisePolynomialParamValue,
    min_anzahl_kinder_für_höheren_freibetrag: int,
    xnp: ModuleType,
) -> float:
    """Use the with-children schedule only from the nth child onwards."""
    erwerbseinkommen_m = (
        einnahmen__bruttolohn_m
        + einkommensteuer__einkünfte__aus_selbstständiger_arbeit__betrag_m
    )
    # Call piecewise_polynomial inside each branch rather than selecting the parameter
    # object first: GETTSIM vectorizes these functions, and a branch that returns a
    # parameter object would be vectorized into an array of them.
    if familie__anzahl_kinder_bis_17_bg >= min_anzahl_kinder_für_höheren_freibetrag:
        out = piecewise_polynomial(
            x=erwerbseinkommen_m,
            parameters=parameter_anrechnungsfreies_einkommen_mit_kindern_in_bg,
            xnp=xnp,
        )
    else:
        out = piecewise_polynomial(
            x=erwerbseinkommen_m,
            parameters=parameter_anrechnungsfreies_einkommen_ohne_kinder_in_bg,
            xnp=xnp,
        )
    return out


reform_zweites_kind = copy_environment(status_quo)
reform_zweites_kind["bürgergeld"]["anrechnungsfreies_einkommen_m"] = (
    anrechnungsfreies_einkommen_m
)
reform_zweites_kind["bürgergeld"]["min_anzahl_kinder_für_höheren_freibetrag"] = (
    ScalarParam(value=2, unit=TTSIMUnit.COUNT.PER_BG)
)

Run that environment on the same input data.

In [ ]:
main(
    main_target=MainTarget.results.df_with_nested_columns,
    policy_date_str=POLICY_DATE,
    policy_environment=reform_zweites_kind,
    input_data=InputData.tree(INPUT_DATA_TREE),
    tt_targets=TTTargets.tree(TARGETS),
    include_warn_nodes=False,
)

---

Keep your reform environment around! Notebook 02 picks the reform environment up
tomorrow.